In [3]:
# ! pip install xgboost

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
df = pd.read_csv('Dataset/phishing_site_urls.csv')

In [5]:
df.head()

,URL,Label
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,bad
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,bad
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,bad
3,mail.printakid.com/www.online.americanexpress....,bad
4,thewhiskeydregs.com/wp-content/themes/widescre...,bad


In [6]:
df.shape


(549346, 2)

In [7]:
df.info()



<class 'pandas.DataFrame'>
RangeIndex: 549346 entries, 0 to 549345
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   URL     549346 non-null  str  
 1   Label   549346 non-null  str  
dtypes: str(2)
memory usage: 8.4 MB


In [8]:
df.isnull().sum()

URL      0
Label    0
dtype: int64

In [9]:
df.Label.value_counts()

Label
good    392924
bad     156422
Name: count, dtype: int64

In [10]:
import re

In [11]:
def tokenize_url(url):
    return re.split(r'\W+', url)

In [12]:
df['tokens'] = df['URL'].apply(tokenize_url)

In [15]:
from gensim.models import Word2Vec

In [16]:
w2v_model = Word2Vec(sentences=df['tokens'], vector_size=100, window=5, min_count=1, workers=4)

In [17]:
def get_embedding(tokens):
    valid_tokens = [t for t in tokens if t in w2v_model.wv]
    if not valid_tokens:
        return np.zeros(w2v_model.vector_size)
    return np.mean([w2v_model.wv[t] for t in valid_tokens], axis=0)

In [18]:
df['embedding'] = df['tokens'].apply(get_embedding)

In [19]:
x = np.vstack(df['embedding'].values)
y = df['Label'].values

In [20]:
df.head()

,URL,Label,tokens,embedding
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,bad,"[nobell, it, 70ffb52d079109dca5664cce6f3173737...","[-0.5978606, -1.2212327, -0.3499424, -1.798703..."
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,bad,"[www, dghjdgf, com, paypal, co, uk, cycgi, bin...","[-0.51108974, -1.2270776, -0.025275338, -1.924..."
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,bad,"[serviciosbys, com, paypal, cgi, bin, get, int...","[-0.69380146, -1.2153304, -0.49232334, -1.9560..."
3,mail.printakid.com/www.online.americanexpress....,bad,"[mail, printakid, com, www, online, americanex...","[-0.03553764, -1.0892463, -1.09201, -2.113584,..."
4,thewhiskeydregs.com/wp-content/themes/widescre...,bad,"[thewhiskeydregs, com, wp, content, themes, wi...","[0.2553867, -1.8951198, -0.19053479, -0.865904..."


### Train traditional Machine learning algorithms ###

In [21]:
from sklearn.model_selection import train_test_split

In [22]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [23]:
import xgboost as xgb

In [24]:
model_xgb = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')

In [28]:
y_train = (y_train == 'good').astype(int)
y_test = (y_test == 'good').astype(int)

In [29]:
model_xgb.fit(x_train, y_train)

C:\Users\Rakshit\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:200: UserWarning: [20:06:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


### LLM Model Training ###

In [1]:
# data = df[['URL', 'Label']]

In [2]:
# data.head()

In [3]:
# data['Label'] = data['Label'].map({'good': 0, 'bad': 1})

In [4]:
# data.rename(columns={'Label': 'label'}, inplace=True)

In [5]:
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
# from datasets import load_dataset, Dataset
# import pandas as pd

In [6]:
# dataset = Dataset.from_pandas(data)

In [7]:
# tokenizer = AutoTokenizer.from_pretrained('roberta-base')
# model = AutoModelForSequenceClassification.from_pretrained('roberta-base', num_labels=2)

In [8]:
# ! pip install torch

In [9]:
# def tokenize(example):
#     return tokenizer(example['URL'], truncation=True, padding='max_length')

In [10]:
# dataset = dataset.map(tokenize, batched=True)
# dataset = dataset.train_test_split(test_size=0.01)

In [11]:
# training_args = TrainingArguments(
#     output_dir="./results",
#     eval_strategy="epoch",
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     num_train_epochs=4,
#     weight_decay=0.01,
# )

In [12]:
# trainer.train()

In [13]:
# model.save_pretrained("phishing_model")
# tokenizer.save_pretrained("phishing_model")

In [14]:
# from transformers import AutoTokenizer, AutoModelForSequenceClassification

# model = AutoModelForSequenceClassification.from_pretrained("phishing_model")
# tokenizer = AutoTokenizer.from_pretrained("phishing_model")

In [15]:
# id2label = {0: 'good', 1: 'bad'}
# pred_labels = [id2label[int(pred)] for pred in predictions]
# print(pred_labels)